# Manager-Worker RL Environment - Training Notebook

This notebook trains a PPO agent on the Manager-Worker RL Environment.

**Environment**: Multi-agent RL where a Manager coordinates Workers to complete tasks under token budget constraints.

**Goal**: Train the Manager to detect and correct worker failures (hallucinations, off-task behavior, incomplete work).

## 1. Setup and Installation

In [ ]:
# Install dependencies
!pip install -q openenv-core gymnasium stable-baselines3 torch wandb tensorboard huggingface-hub pydantic requests

In [ ]:
# Clone repository (if running in Colab)
import os
if not os.path.exists('manager-worker-env'):
    !git clone https://github.com/your-username/openenv-multiagent.git
    os.chdir('openenv-multiagent/manager-worker-env')
else:
    os.chdir('manager-worker-env')

In [ ]:
# Imports
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add current directory to path
sys.path.insert(0, '.')

from env import ManagerWorkerEnv, ManagerAction
from training.train_manager import TrainingConfig, train_manager, EnvironmentWrapper
from training.logging_setup import setup_logging

print("✓ All imports successful")

## 2. Test Environment

In [ ]:
# Create and test environment
config = {
    'max_workers': 4,
    'max_steps': 50,
    'token_budget': 1000,
    'task_difficulty': 3,
    'failure_injection_rate': 0.6,
}

env = ManagerWorkerEnv(config)
obs = env.reset()

print("✓ Environment created and reset")
print(f"  - Task embedding: {len(obs.task_embedding)}-dim")
print(f"  - Worker states: {len(obs.worker_states)}x{len(obs.worker_states[0])}")
print(f"  - Budget: {obs.budget_remaining:.2%}")
print(f"  - Steps: {obs.steps_remaining:.2%}")

In [ ]:
# Run a few random steps
print("Running 5 random steps...")
total_reward = 0
for step in range(5):
    action = ManagerAction(action_id=np.random.randint(0, 7))
    obs, reward, done, info = env.step(action)
    total_reward += reward
    action_name = env.ACTION_NAMES[action.action_id]
    print(f"  Step {step+1}: {action_name:25s} | Reward: {reward:6.2f} | Done: {done}")
    if done:
        break

print(f"\nTotal reward: {total_reward:.2f}")

## 3. Train PPO Agent

In [ ]:
# Create training configuration
training_config = TrainingConfig(
    timesteps=50000,  # Increase for better results
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
)

print("Training Configuration:")
for key, value in training_config.to_dict().items():
    print(f"  {key}: {value}")

In [ ]:
# Train the model
print("Starting training...")
print("This may take a few minutes...\n")

model = train_manager(
    config=training_config,
    model_path='models/ppo_manager_colab',
    verbose=1,
)

print("\n✓ Training complete!")

## 4. Evaluate Trained Model

In [ ]:
# Run trained model on test episodes
from stable_baselines3 import PPO

# Load trained model
model = PPO.load('models/ppo_manager_colab')

print("Running 5 episodes with trained model...\n")

episode_rewards = []
episode_qualities = []
episode_steps = []

for episode in range(5):
    env = ManagerWorkerEnv(config)
    obs = env.reset()
    
    # Convert observation to dict format
    obs_dict = {
        'task_embedding': np.array(obs.task_embedding, dtype=np.float32),
        'worker_states': np.array(obs.worker_states, dtype=np.float32),
        'subtask_status': np.array(obs.subtask_status, dtype=np.int8),
        'budget_remaining': np.array([obs.budget_remaining], dtype=np.float32),
        'steps_remaining': np.array([obs.steps_remaining], dtype=np.float32),
    }
    
    episode_reward = 0
    step_count = 0
    
    while True:
        action, _ = model.predict(obs_dict, deterministic=True)
        manager_action = ManagerAction(action_id=int(action))
        obs, reward, done, info = env.step(manager_action)
        
        obs_dict = {
            'task_embedding': np.array(obs.task_embedding, dtype=np.float32),
            'worker_states': np.array(obs.worker_states, dtype=np.float32),
            'subtask_status': np.array(obs.subtask_status, dtype=np.int8),
            'budget_remaining': np.array([obs.budget_remaining], dtype=np.float32),
            'steps_remaining': np.array([obs.steps_remaining], dtype=np.float32),
        }
        
        episode_reward += reward
        step_count += 1
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    episode_steps.append(step_count)
    
    print(f"Episode {episode+1}: Reward={episode_reward:.2f}, Steps={step_count}")

print(f"\nAverage Reward: {np.mean(episode_rewards):.2f}")
print(f"Average Steps: {np.mean(episode_steps):.1f}")

## 5. Plot Learning Curves

In [ ]:
# Load TensorBoard logs and plot
import pandas as pd
from pathlib import Path

# Try to load TensorBoard logs
log_dir = Path('logs/tensorboard')
if log_dir.exists():
    print("TensorBoard logs found. You can view them with:")
    print(f"  tensorboard --logdir {log_dir}")
else:
    print("No TensorBoard logs found.")

In [ ]:
# Plot episode rewards
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(episode_rewards, marker='o')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Episode Rewards (Trained Model)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(episode_steps, marker='s')
plt.xlabel('Episode')
plt.ylabel('Steps to Complete')
plt.title('Episode Length (Trained Model)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Plots generated")

## 6. Save Model to HuggingFace Hub (Optional)

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and fill in your details to push the model

# from training.huggingface_integration import HuggingFaceIntegration
# 
# hf_integration = HuggingFaceIntegration(repo_id='your-username/manager-worker-rl')
# hf_integration.push_model(
#     model_path='models/ppo_manager_colab',
#     training_timesteps=50000,
#     hyperparameters=training_config.to_dict(),
# )

print("To push to HuggingFace Hub:")
print("1. Uncomment the code above")
print("2. Replace 'your-username' with your HuggingFace username")
print("3. Run the cell")

## 7. Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nEnvironment: Manager-Worker RL")
print(f"Training Timesteps: {training_config.timesteps:,}")
print(f"Learning Rate: {training_config.learning_rate}")
print(f"\nModel Performance:")
print(f"  Average Reward: {np.mean(episode_rewards):.2f}")
print(f"  Average Steps: {np.mean(episode_steps):.1f}")
print(f"  Max Reward: {np.max(episode_rewards):.2f}")
print(f"  Min Reward: {np.min(episode_rewards):.2f}")
print(f"\nModel saved to: models/ppo_manager_colab.zip")
print(f"\nNext steps:")
print(f"  1. Download the model from Colab")
print(f"  2. Push to HuggingFace Hub (see cell above)")
print(f"  3. Deploy to HuggingFace Spaces")
print("="*60)